In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
# Widgets (fallback local)
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

In [0]:
silver_df = spark.read.table("bronze.transactions_sales")

In [0]:
display(silver_df)

In [0]:
silver_df = silver_df.dropDuplicates()

In [0]:
silver_df.withColumn("data_venda", F.col("data_venda").cast(StringType()))

In [0]:
silver_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_df.columns
]).show()

In [0]:
for field in silver_df.schema.fields:
    if isinstance(field.dataType, StringType):
        silver_df = silver_df.withColumn(field.name,F.trim(F.col(field.name)))

In [0]:
silver_df = (
    silver_df
    .withColumn("preco_unitario", F.col("preco_unitario").cast(DecimalType(10, 2) ))
    .withColumn("custo_unitario", F.col("custo_unitario").cast(DecimalType(10, 2) ))
)

In [0]:
silver_df = (
    silver_df
    .withColumn("qtd_vendida",  F.col("qtd_vendida").cast(IntegerType()))
    .withColumn("data_venda", F.to_timestamp(F.col("data_venda"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("data_venda", F.to_date(F.col("data_venda"), "yyyy-MM-dd"))
    .withColumn("nome", F.split(F.col("nome_cliente"), ", ").getItem(0))
    .withColumn("sobrenome", F.split(F.col("nome_cliente"), ", ").getItem(1))
    .withColumn("pais", F.split(F.col("localidade"), " - ").getItem(0))
    .withColumn("continente", F.split(F.col("localidade"), " - ").getItem(1))
    .drop("nome_cliente", "localidade", "ingestion_time")
)

display(silver_df)

In [0]:
# Validação de dados
silver_df = (
    silver_df
    .filter(F.col("data_venda").isNotNull())
    .filter(F.col("preco_unitario") > 0)
    .filter(F.col("custo_unitario") > 0)
    .filter(F.col("qtd_vendida") > 0)
)

In [0]:
silver_df.printSchema()

In [0]:
silver_df.write.mode("overwrite") \
    .saveAsTable("silver.clean_sales")

print("Dados gravados com sucesso")